# 3.5 — Cross-Validation

A single train/test split can be lucky or unlucky depending on which rows ended up in the test set. Cross-validation repeats the evaluation multiple times on different splits — giving you a reliable, trustworthy score.

### The problem with a single split:
```
Split 1 (random_state=42):  accuracy = 0.81  ← you report this
Split 2 (random_state=99):  accuracy = 0.76  ← very different!
Split 3 (random_state=7):   accuracy = 0.84  ← also different!
```
Which number is real? None of them alone. The average across many splits is.

### How 5-fold cross-validation works:
```
Fold 1: [TEST ][train][train][train][train] → score
Fold 2: [train][TEST ][train][train][train] → score
Fold 3: [train][train][TEST ][train][train] → score
Fold 4: [train][train][train][TEST ][train] → score
Fold 5: [train][train][train][train][TEST ] → score

Final = mean ± std of all 5 scores
```

> **Always report: mean ± std — not just a single accuracy number.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Load and prepare Titanic
df = sns.load_dataset('titanic')
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

X = df.drop(columns=['survived'])
y = df['survived']

# Note: for cross_val_score we pass unscaled X
# scaling happens inside each fold automatically when using Pipeline (chapter 7)
# for now, scale the full X for simplicity
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Setup complete. Dataset shape:", X.shape)

## The Problem — Single Split is Unreliable

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)

# Same model, different random splits — see how accuracy varies
print("Same model, different train/test splits:")
for seed in [0, 7, 42, 99, 123]:
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=seed)
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"  random_state={seed:>3}: accuracy = {acc:.3f}")

print("\nThe scores vary — which one do you trust?")

## Basic Cross-Validation

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)

# 5-fold cross-validation
scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')

print("5-fold CV scores:", scores.round(3))
print(f"Mean:  {scores.mean():.3f}")
print(f"Std:   {scores.std():.3f}")
print(f"\nReport as: {scores.mean():.3f} ± {scores.std():.3f}")
print("\nSmall std = consistent model (good)")
print("Large std = unstable model (bad — results vary too much)")

## Cross-Validation with Different Metrics

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)

for metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring=metric)
    print(f"{metric:12s}: {scores.mean():.3f} ± {scores.std():.3f}")

## Stratified K-Fold — Better for Imbalanced Data

Regular KFold splits randomly. StratifiedKFold ensures each fold has the same class ratio as the full dataset.
Always use StratifiedKFold for classification.

In [ ]:
# Regular KFold vs StratifiedKFold
model = LogisticRegression(random_state=42, max_iter=1000)

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_kf  = cross_val_score(model, X_scaled, y, cv=kf,  scoring='f1')
scores_skf = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1')

print(f"KFold F1:           {scores_kf.mean():.3f} ± {scores_kf.std():.3f}")
print(f"StratifiedKFold F1: {scores_skf.mean():.3f} ± {scores_skf.std():.3f}")
print("\nStratifiedKFold gives more reliable score for imbalanced data")

# Show class distribution in each fold
print("\nClass ratio per fold (StratifiedKFold):")
for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), 1):
    ratio = y.iloc[test_idx].mean()
    print(f"  Fold {fold}: {ratio:.2f} survival rate")

## Comparing Models With Cross-Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':25s}  {'Accuracy':>10}  {'F1':>10}  {'AUC':>10}")
print("-" * 62)

for name, m in models.items():
    acc = cross_val_score(m, X_scaled, y, cv=skf, scoring='accuracy')
    f1  = cross_val_score(m, X_scaled, y, cv=skf, scoring='f1')
    auc = cross_val_score(m, X_scaled, y, cv=skf, scoring='roc_auc')
    print(f"{name:25s}  {acc.mean():.3f}±{acc.std():.2f}  {f1.mean():.3f}±{f1.std():.2f}  {auc.mean():.3f}±{auc.std():.2f}")

## Visualise CV Score Distribution

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(model, X_scaled, y, cv=skf, scoring='f1')

plt.figure(figsize=(8, 4))
plt.bar(range(1, 11), scores, color='#378ADD', alpha=0.7)
plt.axhline(scores.mean(), color='#E24B4A', linewidth=2, linestyle='--',
            label=f'Mean = {scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.title('10-fold CV F1 Scores — Logistic Regression')
plt.ylim(0.5, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean F1: {scores.mean():.3f}")
print(f"Std:     {scores.std():.3f}")
print(f"Min:     {scores.min():.3f}")
print(f"Max:     {scores.max():.3f}")

## Key Rules

| Rule | Why |
|------|-----|
| Always use CV instead of single split | More reliable, less lucky/unlucky |
| Report mean ± std | Shows consistency, not just average |
| Use StratifiedKFold for classification | Preserves class ratio per fold |
| Use 5 or 10 folds | 5 is standard, 10 for more reliable estimate |
| Large std = unstable model | Model is sensitive to which data it sees |

> **Key insight:** A model with accuracy 0.83 ± 0.01 is far more trustworthy than a model with 0.86 ± 0.08. Consistency matters as much as the score itself.

## Practice Task

In [ ]:
# Q1 — Run 5-fold CV on Logistic Regression. Report mean ± std for accuracy and F1.
# YOUR CODE HERE

# Q2 — Run the same CV on a DecisionTreeClassifier. Which model is better?
# YOUR CODE HERE

# Q3 — Use StratifiedKFold with 10 folds. Does the mean change much vs 5 folds?
# YOUR CODE HERE

# Q4 — What does a large std in CV scores tell you about a model?
# ANSWER:

# Q5 — Why is StratifiedKFold better than regular KFold for Titanic?
# ANSWER: